In [5]:
from alerce.core import Alerce
from astropy.table import vstack, Table
import json
import matplotlib.pyplot as plt
import pyvo as vo
import requests
import sqlalchemy as sa
import sys
import numpy as np
from astropy.coordinates import SkyCoord
from astropy import units as u
import re
import george
import scipy.optimize as op
import emcee
from matplotlib.backends.backend_pdf import PdfPages

In [6]:
alerce = Alerce()
alerce_tap = vo.dal.TAPService('https://tap.alerce.online/tap')

In [7]:
filename = "../Data/tns_Ia.csv"
savefile = "../Data/" + filename[12:14] + ".json"
sources = Table.read(filename, format="csv")
savefile

'../Data/Ia.json'

In [10]:
ZTF_mask = sources["source_group"] == "ZTF"
ZTF = sources[ZTF_mask]

In [9]:
#ZTF["internal_names"][0][:12]
ZTF_obj = []
pattern = r"ZTF"
for obj in ZTF["internal_names"]:
    #print(obj)
    if isinstance(obj, str):
        internal = obj.split(',')
        survay_name = [name for name in internal if re.search(pattern, name)]
        if survay_name:
            ZTF_obj.append(survay_name[0])

ZTF_obj = np.array(ZTF_obj)
print(len(ZTF_obj))

6425


In [9]:
max_curves = 20
light_curves = []
for obj in ZTF_obj[:max_curves]:
    light = alerce.query_forced_photometry(obj, format="json")
    light_curves.append(light)

In [28]:
with open(savefile, "w") as file:
    json.dump(light_curves, file)